# SMART HOSTEL ATTENDANCE — FACENET FACIAL RECOGNITION EXPERIMENTATION NOTEBOOK

**Project**: Smart Hostel Attendance Management System Using Face Recognition and Geolocation  
**Purpose**: Experimental validation & testing of the FaceNet (InceptionResnetV1) + MTCNN pipeline on Kaggle.  

> **Important Architectural Notes**:
> 1. **No Model Training**: We use the pretrained `InceptionResnetV1` model (trained on VGGFace2) directly for zero-shot 512-d feature extraction. No custom CNN training is required.
> 2. **No OpenCV**: OpenCV (`cv2`) is strictly avoided; image processing is performed via `Pillow (PIL)` and `PyTorch`.
> 3. **Kaggle Scope**: This notebook is purely for algorithm experimentation and testing. The production application runs via the Python microservice (`face-service/app.py`) communicating with the Java Spring Boot backend.

## Step 1: Install & Import Required Packages

In [ ]:
# Install facenet-pytorch (PyTorch and Pillow are pre-installed in Kaggle environments)
!pip install facenet-pytorch

import io
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw
from facenet_pytorch import MTCNN, InceptionResnetV1

# Check CUDA GPU acceleration
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Using computation device: {device}")

## Step 2 & 3: Load Pretrained MTCNN Detector & FaceNet Model

In [ ]:
# 1. Initialize MTCNN for face detection and 160x160 alignment
# keep_all=True is used to detect if multiple faces are present
mtcnn = MTCNN(
    image_size=160,
    margin=20,
    min_face_size=20,
    thresholds=[0.6, 0.7, 0.7],
    factor=0.709,
    post_process=True,
    keep_all=True,
    device=device
)

# 2. Load Pretrained InceptionResnetV1 FaceNet model (VGGFace2)
facenet = InceptionResnetV1(pretrained='vggface2').eval().to(device)
print("Pretrained FaceNet and MTCNN models initialized successfully.")

## Step 4, 5, 6 & 7: Face Detection, Validation, and Embedding Generation

In [ ]:
def extract_embedding(image_input):
    """
    Accepts PIL Image, detects face with MTCNN, enforces Single Face Policy,
    and extracts a 512-dimensional L2-normalized FaceNet embedding.
    """
    if not isinstance(image_input, Image.Image):
        image = Image.open(image_input).convert('RGB')
    else:
        image = image_input.convert('RGB')

    # Detect faces
    boxes, probs = mtcnn.detect(image)

    # Validation rule: Reject if 0 faces
    if boxes is None or len(boxes) == 0:
        return {"valid": False, "reason": "Face not detected. Please upload another photo.", "embedding": None}

    # Filter high-confidence detections
    valid_boxes = [b for i, b in enumerate(boxes) if probs[i] is not None and probs[i] >= 0.85]
    face_count = len(valid_boxes)

    if face_count == 0:
        return {"valid": False, "reason": "Face not detected. Please upload another photo.", "embedding": None}

    # Validation rule: Reject if >1 faces
    if face_count > 1:
        return {"valid": False, "reason": "Multiple faces detected. Please upload a photo containing only you.", "embedding": None}

    # Extract aligned 160x160 face tensor
    face_tensor = mtcnn(image)
    if face_tensor is None:
        return {"valid": False, "reason": "Unable to process this image.", "embedding": None}

    if face_tensor.ndim == 4:
        face_tensor = face_tensor[0]

    face_tensor = face_tensor.unsqueeze(0).to(device)

    # Generate 512-d FaceNet Embedding with L2 Normalization
    with torch.no_grad():
        raw_embedding = facenet(face_tensor)
        norm_embedding = F.normalize(raw_embedding, p=2, dim=1)
        embedding_np = norm_embedding.squeeze(0).cpu().numpy()

    return {"valid": True, "reason": "Valid Face Profile", "embedding": embedding_np}

## Step 8: Cosine Similarity Helper & Multi-Photo Comparison Rule

In [ ]:
def cosine_similarity(v1, v2):
    """Calculates cosine similarity between two 512-d vectors."""
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return float(np.dot(v1, v2) / (norm1 * norm2))

FACE_MATCH_THRESHOLD = 0.70  # Configurable threshold

## Step 9, 10, 11 & 12: End-to-End Simulation with Registered Photos vs Live Photo

In [ ]:
# Create synthetic test faces (or load real test images from file)
def create_test_face_canvas(seed=42):
    np.random.seed(seed)
    img = Image.new('RGB', (200, 200), color=(240, 230, 215))
    draw = ImageDraw.Draw(img)
    # Draw head
    draw.ellipse([40, 30, 160, 170], fill=(220, 190, 160), outline=(139, 107, 74))
    # Draw eyes
    draw.ellipse([65, 75, 85, 95], fill=(63, 48, 37))
    draw.ellipse([115, 75, 135, 95], fill=(63, 48, 37))
    # Draw nose & mouth
    draw.line([100, 95, 100, 120], fill=(139, 107, 74), width=3)
    draw.arc([75, 125, 125, 145], 0, 180, fill=(185, 102, 78), width=3)
    return img

# 1. Enroll up to 10 photos for student ASIET2024CS001
enrolled_photos = [create_test_face_canvas(seed=100 + i) for i in range(5)]
registered_embeddings = []

print("=== ENROLLING STUDENT PHOTOS (Max 10) ===")
for idx, photo in enumerate(enrolled_photos):
    res = extract_embedding(photo)
    if res["valid"]:
        registered_embeddings.append(res["embedding"])
        print(f"Photo #{idx+1}: ACCEPTED (512-d embedding generated)")
    else:
        print(f"Photo #{idx+1}: REJECTED - {res['reason']}")

print(f"Total Registered Templates: {len(registered_embeddings)} / 10\n")

# 2. Test with Matching Live Photo
live_photo_match = create_test_face_canvas(seed=102) # Similar to enrolled Photo #3
live_res_match = extract_embedding(live_photo_match)

print("=== TESTING LIVE FACE VERIFICATION (MATCH TEST) ===")
if live_res_match["valid"]:
    live_emb = live_res_match["embedding"]
    scores = [cosine_similarity(live_emb, reg) for reg in registered_embeddings]
    max_sim = max(scores) if scores else 0.0
    best_photo = scores.index(max_sim) + 1 if scores else 0
    matched = max_sim >= FACE_MATCH_THRESHOLD

    print(f"All Photo Similarities: {[round(s, 4) for s in scores]}")
    print(f"Max Similarity Score: {max_sim:.4f} (Threshold: {FACE_MATCH_THRESHOLD})")
    print(f"Best Matching Template: Photo #{best_photo}")
    print(f"Result: {'MATCH' if matched else 'NO MATCH'}")
else:
    print(f"Live Photo Rejected: {live_res_match['reason']}")

# 3. Test with Non-Matching Live Photo
print("\n=== TESTING LIVE FACE VERIFICATION (NON-MATCH TEST) ===")
live_photo_impostor = create_test_face_canvas(seed=999)
live_res_impostor = extract_embedding(live_photo_impostor)

if live_res_impostor["valid"]:
    live_emb_imp = live_res_impostor["embedding"]
    scores_imp = [cosine_similarity(live_emb_imp, reg) for reg in registered_embeddings]
    max_sim_imp = max(scores_imp) if scores_imp else 0.0
    matched_imp = max_sim_imp >= FACE_MATCH_THRESHOLD

    print(f"Max Similarity Score: {max_sim_imp:.4f} (Threshold: {FACE_MATCH_THRESHOLD})")
    print(f"Result: {'MATCH' if matched_imp else 'NO MATCH'}")